# ResNet18 Fine-Tuning — FER-2013 Emotion Recognition

Pipeline: Webcam → YOLO pose → face crop → ResNet emotion classification

Dieses Notebook trainiert ResNet18 auf FER-2013 (7 Emotionen) und exportiert die Gewichte.

In [ ]:
# Zelle 1: Setup & Imports
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import tarfile
import csv
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Zelle 2: FER-2013 downloaden & entpacken
# Quelle: TensorFlow Datasets Mirror (Google Cloud Storage)
!wget -q https://storage.googleapis.com/tensorflow/tf-keras-datasets/fer2013.tar.gz -O fer2013.tar.gz
!tar -xzf fer2013.tar.gz
print("fer2013.csv heruntergeladen")
!head -3 fer2013.csv

In [ ]:
# Zelle 3: CSV → ImageFolder-Struktur konvertieren
# FER-2013 hat Spalten: emotion, pixels, Usage
# Usage = Training | PublicTest | PrivateTest
# PublicTest + PrivateTest → test, Training → train

EMOTIONEN = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

for split in ["train", "test"]:
    for emotion in EMOTIONEN:
        os.makedirs(f"data/raw/{split}/{emotion}", exist_ok=True)

with open("fer2013.csv") as f:
    reader = csv.reader(f)
    next(reader)  # header überspringen
    for idx, (emotion, pixels, usage) in enumerate(reader):
        split = "test" if usage in ("PublicTest", "PrivateTest") else "train"
        emotion_name = EMOTIONEN[int(emotion)]
        pixel_values = np.array(pixels.split(), dtype=np.uint8).reshape(48, 48)
        img = Image.fromarray(pixel_values)
        img.save(f"data/raw/{split}/{emotion_name}/fer{idx}.png")

print("Konvertierung fertig!")
!find data/raw -name "*.png" | wc -l
!ls data/raw/train/

In [ ]:
# Zelle 4: Dataset & DataLoader
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("data/raw/train", transform=transform)
test_dataset = datasets.ImageFolder("data/raw/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} Bilder, {len(train_loader)} Batches")
print(f"Test:  {len(test_dataset)} Bilder, {len(test_loader)} Batches")
print(f"Klassen (alphabetisch): {train_dataset.classes}")

In [ ]:
# Zelle 5: Batch visualisieren
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    # Denormalisieren für Anzeige
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(train_dataset.classes[labels[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 6: ResNet18 mit ImageNet-Gewichten laden
NUM_CLASSES = 7
LR = 0.001

model = models.resnet18(weights="DEFAULT")
model.fc = nn.Linear(512, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Modell auf: {next(model.parameters()).device}")
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Zelle 7: Training-Loop
EPOCHS = 5
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoche {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

In [ ]:
# Zelle 8: Loss-Verlauf plotten
plt.plot(range(1, EPOCHS+1), train_losses, marker='o', linewidth=2)
plt.title("Training Loss over Epochs")
plt.xlabel("Epoche")
plt.ylabel("CrossEntropy Loss")
plt.grid(True)
plt.show()

In [ ]:
# Zelle 9: Validation — Accuracy auf Test-Set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Test-Accuracy: {accuracy:.2f}% ({correct}/{total})")

In [ ]:
# Zelle 10: Konfusionsmatrix
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
class_names = ["Wut", "Ekel", "Angst", "Freude", "Neutral", "Trauer", "Überraschung"]

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — FER-2013 Test Set")
plt.tight_layout()
plt.show()

# Pro-Klasse Accuracy
print("\nPer-Class Accuracy:")
for i, name in enumerate(class_names):
    total_cls = (np.array(all_labels) == i).sum()
    correct_cls = (np.array(all_labels)[np.array(all_preds) == i] == i).sum() if total_cls > 0 else 0
    acc_cls = (np.array(all_preds)[np.array(all_labels) == i] == i).sum() / total_cls * 100
    print(f"  {name:12s}: {acc_cls:.1f}% ({int(correct_cls)}/{total_cls})")

In [ ]:
# Zelle 11: Modell exportieren & runterladen
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/resnet_fer2013.pth")
print("Modell gespeichert: models/resnet_fer2013.pth")

from google.colab import files
files.download("models/resnet_fer2013.pth")